In [50]:
from langgraph.graph import StateGraph, START , END
from langchain_groq import ChatGroq
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [51]:
load_dotenv()

True

In [52]:
model=ChatGroq(model='openai/gpt-oss-120b')

In [53]:
#this schnema for the sentiment to extract sentiment from the review
class sentimentschema(BaseModel):
    sentiment:Literal['negative', 'positive'] = Field(description='sentiment of the review')

#this schema for the daignosis of review analysis of review to give every thing in jason object
class DaignosisSchema(BaseModel):
    issue_type: Literal["UI", "Performance", "Bug", "Support", "Other"] = Field(
description="The category of issue mentioned in the review."
)
    tonality: Literal["Frustrated", "Angry", "Disappointed", "Confused"] = Field(
description="The emotional tone expressed by the user."
)
    urgency: Literal["Low", "Medium", "High"] = Field(
description="How urgent or critical the issue appears to be."
)
    

In [54]:
structured_model=model.with_structured_output(sentimentschema)
structured_model2=model.with_structured_output(DaignosisSchema)

In [55]:
class reviewState(TypedDict):
    review: str
    sentiment:Literal['negative',  'positive']
    daignosis: dict
    response:str

In [56]:
def find_sent(state:reviewState)->reviewState:

    review=state['review']

    prompt=f"find sentiment of the following review - {review}"
    sentiment=structured_model.invoke(prompt).sentiment

    return {
        'sentiment':sentiment
        
    }

def check_condition(state:reviewState)-> Literal['positive_response', 'run_daignosis']:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_daignosis'


In [ ]:
def positive_response(state:reviewState)->reviewState:

    prompt=f"Write a warm thank-you message in response to this review:\n\n\"{state['review']}\"\n. Also, kindly ask the user to leave feedback on our website"
    response= model.invoke(prompt).content
    return {
        'response':response
    }

def run_daignosis(state: reviewState)->reviewState:
    review=state['review']

    prompt=f"""daignosis this negative review :\n\n {review}\n -> return issue type, tone and urgency"""
    response= structured_model2.invoke(prompt)

    return {'daignosis': response.model_dump()} #convert from json to dictionary


def negative_response(state: reviewState)->reviewState:
# Extract diagnosis details from the state
    diagnosis = state['daignosis']
    issue_type = diagnosis['issue_type']
    tone = diagnosis['tonality']
    urgency = diagnosis.get('urgency')
    review = state['review'] 

# Prepare the prompt with extracted details
    prompt = f"""You are a support assistant.
The user had a {issue_type} issue.
The review sounded {tone} and marked urgency as {urgency}.
Write an empathetic, helpful resolution message to this review: {review}"""

# Generate response using the model
    response = model.invoke(prompt).content
    return {"response": response.content}
    


In [58]:
graph=StateGraph(reviewState)
graph.add_node('find_sent', find_sent)
graph.add_node('positive_response', positive_response)
graph.add_node('run_daignosis', run_daignosis)
graph.add_node('negative_response', negative_response)

#edges
graph.add_edge(START, 'find_sent')
graph.add_conditional_edges(
    'find_sent',
    check_condition,
    {
        'positive_response':'positive_response',
        'run_daignosis':'run_daignosis'
    }
)
graph.add_edge('positive_response',END)
graph.add_edge('run_daignosis','negative_response')
graph.add_edge('negative_response',END )


In [59]:
workflow=graph.compile()

In [61]:
initial_state={
    'review':"I have been trying to log in for over an hour now. And the app keeps freezing on the authentication screen. I even tried reinstalling it. But no luck. This kind of bug is unacceptable especially when it affects basic functionality."
}
workflow.invoke(initial_state)

{'review': 'I have been trying to log in for over an hour now. And the app keeps freezing on the authentication screen. I even tried reinstalling it. But no luck. This kind of bug is unacceptable especially when it affects basic functionality.',
 'sentiment': 'negative',
 'daignosis': {'issue_type': 'Bug',
  'tonality': 'Frustrated',
  'urgency': 'High'},
 'response': 'Hi\u202f[Your Name],\n\nI’m really sorry you’re stuck on the authentication screen—having to wait an hour and even reinstall the app with no result is absolutely frustrating, and I understand why you’d feel this is unacceptable. Let’s get you back in as quickly as possible.\n\n**Here are a few steps that often resolve this freezing issue:**\n\n1. **Clear the app’s cache & data**  \n   - **iOS:**\u202fSettings → General → iPhone Storage → *Your App* → “Offload App” (this removes the app but keeps its data) and then reinstall.  \n   - **Android:**\u202fSettings → Apps → *Your App* → Storage → “Clear Cache” and “Clear Data,